In [6]:
!pip install -U datasets huggingface_hub fsspec

  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import math
import datasets
import requests

## Dataset and Tokenizer


### Dataset download

In [8]:
dataset = datasets.load_dataset("wmt/wmt14", "de-en", split=["train", "test"])

In [10]:
dataset

[Dataset({
     features: ['translation'],
     num_rows: 4508785
 }),
 Dataset({
     features: ['translation'],
     num_rows: 3003
 })]

In [4]:
df = pd.DataFrame(dataset[0]["translation"])

In [ ]:
df = df.loc[:500000]

In [19]:
unk_idx = 0
pad_idx = 1
sos_idx = 2
eos_idx = 3

### Train Tokenizer

In [20]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace, Punctuation, Sequence as Pre_Seq
from tokenizers.normalizers import Lowercase, Sequence
from tokenizers.processors import TemplateProcessing
from tokenizers.normalizers import Sequence, NFC, Lowercase
from copy import deepcopy

tkz_en = Tokenizer(BPE(unk_token="[UNK]"))
tkz_en.normalizer = Sequence([NFC(), Lowercase()])
tkz_en.pre_tokenizer = Pre_Seq([Whitespace(), Punctuation()])
tkz_en.post_processor = TemplateProcessing(
    single="[SOS] $A [EOS]",
    special_tokens=[("[SOS]", sos_idx), ("[EOS]", eos_idx)],
)
tkz_de = deepcopy(tkz_en)
trainer = BpeTrainer(vocab_size=30000, special_tokens=["[UNK]", "[PAD]", "[SOS]", "[EOS]"])

In [7]:
training_data_de = df["de"].to_list()
training_data_en = df["en"].to_list()
import numpy as np
np.random.shuffle(training_data_de)
np.random.shuffle(training_data_en)

In [21]:
tkz_en.train_from_iterator(training_data_en, trainer)
tkz_de.train_from_iterator(training_data_de, trainer)

NameError: name 'training_data_en' is not defined

In [9]:
tkz_en.save("tokenizer_en.json")
tkz_de.save("tokenizer_de.json")

### Load Tokenizer

In [ ]:
text = requests.get("https://raw.githubusercontent.com/TheGEN1U5/DLventures/refs/heads/main/nlp/seq2seq/tokenizer_en.json").text
with open("tokenizer_en.json", "w") as f:
    f.write(text)
text = requests.get("https://raw.githubusercontent.com/TheGEN1U5/DLventures/refs/heads/main/nlp/seq2seq/tokenizer_de.json").text
with open("tokenizer_de.json", "w") as f:
    f.write(text)

In [22]:
tkz_en = Tokenizer.from_file("tokenizer_en.json")
tkz_de = Tokenizer.from_file("tokenizer_de.json")

In [23]:
tkz_en.encode("stupidly").ids
tkz_de.encode("mein kampf").ids

[2, 3558, 4753, 3]

### Tokenize Data

In [26]:
df["de_tokens"] = df["de"].apply(lambda x: torch.tensor(tkz_de.encode(x).ids, dtype=torch.long))
df["en_tokens"] = df["en"].apply(lambda x: torch.tensor(tkz_en.encode(x).ids, dtype=torch.long))

df["de_len"] = df["de_tokens"].apply(lambda x: len(x))
df["en_len"] = df["en_tokens"].apply(lambda x: len(x))

In [27]:
df.to_csv("df.csv", index=False)

In [28]:
import pandas as pd
df = pd.read_csv("df.csv")
len(df)

500000

### Sampling Function

In [ ]:
from sklearn.model_selection import train_test_split
def sample(df, size, val_split):
    num_bins = 10  # increase/decrease depending on precision vs sample size tradeoff
    df['bin'] = pd.cut(df['en_len'], bins=num_bins)
    sampled_df = (
        df.groupby('bin', group_keys=False)
        .apply(lambda x: x.sample(frac=size / len(df), random_state=42))
    )
    train_df, val_df = train_test_split(
        sampled_df,
        test_size=val_split,  # 20% goes to val+test
        stratify=sampled_df['bin'],
        random_state=42
    )
    return train_df, val_df

In [ ]:
train_df, val_df = sample(df, 500000, 0.1)

/tmp/ipykernel_2697/3875357300.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('bin', group_keys=False)


ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

### Create Loaders

In [24]:
def collate_fn(batch):
    en, de = zip(*batch)
    padded_en = pad_sequence(en, batch_first=True, padding_value=pad_idx)
    padded_de = pad_sequence(de, batch_first=True, padding_value=pad_idx)
    return padded_en, padded_de

In [25]:
def get_loader(df, batch_size):
    return DataLoader(list(zip(df["en_tokens"].to_numpy(), df["de_tokens"].to_numpy())), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

## Transformer Implementation

In [26]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_seq_len, embed_dim):
        super(PositionalEmbedding, self).__init__()
        self.embed_dim = embed_dim
        pos_mask = torch.zeros((max_seq_len, embed_dim))
        for pos in range(max_seq_len):
            for i in range(0, self.embed_dim, 2):
                pos_mask[pos, i] = np.sin(pos / 10000 ** (i / self.embed_dim))
                pos_mask[pos, i + 1] = np.cos(pos / 10000 ** (i / self.embed_dim))

        pos_mask = pos_mask.unsqueeze(0)
        self.register_buffer("pos_mask", pos_mask)

    def forward(self, x):
        seq_len = x.size(1)
        x = x + torch.autograd.Variable(self.pos_mask[:,:seq_len], requires_grad=False)
        return x

In [27]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        self.e = embed_dim // num_heads
        self.d_k = key_dim
        self.d_v = value_dim
        self.k_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.q_proj = nn.ModuleList([nn.Linear(self.e, self.d_k) for _ in range(num_heads)])
        self.v_proj = nn.ModuleList([nn.Linear(self.e, self.d_v) for _ in range(num_heads)])
        self.out = nn.Linear(self.num_heads * self.d_v, self.embed_dim)

    def forward(self, key, query, value, mask=None):
        self.mask = mask

        batch_size, seq_len = key.size(0), key.size(1)
        key = key.reshape(batch_size, seq_len, self.num_heads, self.e)
        query = query.reshape(batch_size, seq_len, self.num_heads, self.e)
        value = value.reshape(batch_size, seq_len, self.num_heads, self.e)

        k = torch.stack([proj(key[:, :, i, :]) for i, proj in enumerate(self.k_proj)], dim=2)  # [n, l, h, d_k]
        q = torch.stack([proj(query[:, :, i, :]) for i, proj in enumerate(self.q_proj)], dim=2)
        v = torch.stack([proj(value[:, :, i, :]) for i, proj in enumerate(self.v_proj)], dim=2)

        qkt_scaled = torch.einsum("nqhd,nkhd->nhqk", q, k) / math.sqrt(self.d_k)
        if mask is not None:
            qkt_scaled = qkt_scaled.masked_fill(mask == 0, float("-1e20"))
        sftmx = torch.softmax(qkt_scaled, dim=-1)

        scores = torch.einsum("nhqk,nkhd->nqhd", sftmx, v)
        concat = scores.reshape(scores.size(0), scores.size(1), -1)

        output = self.out(concat)
        return output

In [28]:
class Encoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Encoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)

    def forward(self, x):
        mha_output = self.MHA(x, x, x)
        add_norm1 = self.norm1(mha_output + self.drop1(x))
        ffn_output = self.ffn(add_norm1)
        output = self.norm2(ffn_output + self.drop2(add_norm1))
        return output

In [29]:
class Decoder(nn.Module):
    def __init__(self, embed_dim, key_dim, value_dim, num_heads):
        super(Decoder, self).__init__()
        self.embed_dim = embed_dim
        self.d_k = key_dim
        self.d_v = value_dim
        self.num_heads = num_heads

        self.masked_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.cross_MHA = MultiHeadAttention(self.embed_dim, self.d_k, self.d_v, self.num_heads)
        self.norm2 = nn.LayerNorm(self.embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, 2 * self.embed_dim),
            nn.ReLU(),
            nn.Linear(2 * self.embed_dim, self.embed_dim)
        )
        self.norm3 = nn.LayerNorm(self.embed_dim)
        self.drop1 = nn.Dropout(0.1)
        self.drop2 = nn.Dropout(0.1)
        self.drop3 = nn.Dropout(0.1)

    def forward(self, x, enc_output):
        batch_size, trg_len, _ = x.shape
        # returns the lower triangular part of matrix filled with ones
        mask = torch.tril(torch.ones((trg_len, trg_len))).expand(
            batch_size, 1, trg_len, trg_len
        )
        masked_mha_output = self.masked_MHA(x, x, x, mask=mask)
        add_norm1 = self.norm1(masked_mha_output + self.drop1(x))
        cross_mha_output = self.cross_MHA(enc_output, add_norm1, enc_output)
        add_norm2 = self.norm2(cross_mha_output + self.drop2(add_norm1))
        ffn_output = self.ffn(add_norm2)
        output = self.norm3(ffn_output + self.drop3(add_norm2))
        return output

In [31]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, vocab_size_src, vocab_size_tgt, key_dim, value_dim, num_heads, num_encoder_layers=1, num_decoder_layers=1, max_seq_len=100):
        super(Transformer, self).__init__()
        self.embedding_src = nn.Embedding(vocab_size_src, embed_dim, padding_idx=1)
        self.embedding_trg = nn.Embedding(vocab_size_tgt, embed_dim, padding_idx=1)
        self.pe = PositionalEmbedding(max_seq_len, embed_dim)
        self.encoder_layers = nn.ModuleList([
            Encoder(embed_dim, key_dim, value_dim, num_heads) for _ in range(num_encoder_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            Decoder(embed_dim, key_dim, value_dim, num_heads) for _ in range(num_decoder_layers)
        ])
        self.out = nn.Linear(embed_dim, vocab_size_tgt)

    def forward(self, x, y):
        v_x = self.embedding_src(x)
        v_x = self.pe(v_x)
        v_y = self.embedding_trg(y)
        v_y = self.pe(v_y)
        src = v_x
        tgt = v_y
        for layer in self.encoder_layers:
            src = layer(src)
        for layer in self.decoder_layers:
            tgt = layer(tgt, src)
        logits = self.out(tgt)
        probs = torch.softmax(logits, dim=-1)
        return probs

### Training loop

In [32]:
def train(model, device, train_loader, val_loader, num_epochs=10, learning_rate=1e-3, pad_idx=0, early_stop=False):
    model.to(device)

    criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, betas=(0.9, 0.98), eps=1e-9)

    train_losses = []
    val_losses = []

    best_val_loss = float('inf')
    patience_counter = 0
    patience_limit = 10
    best_model_state = None

    for epoch in range(num_epochs):
        # Training
        model.train()
        total_loss = 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src = src.to(device)
            tgt = tgt.to(device)

            tgt_input = tgt[:, :-1]
            tgt_output = tgt[:, 1:]

            logits = model(src, tgt_input)

            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
            optimizer.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        # Validation
        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for src, tgt in val_loader:
                src = src.to(device)
                tgt = tgt.to(device)

                tgt_input = tgt[:, :-1]
                tgt_output = tgt[:, 1:]

                logits = model(src, tgt_input)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output.reshape(-1))
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        # Early stopping logic
        if early_stop:
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_model_state = model.state_dict()
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= patience_limit:
                    if best_model_state is not None:
                        model.load_state_dict(best_model_state)
                    return train_losses, val_losses

    if early_stop and best_model_state is not None:
        model.load_state_dict(best_model_state)

    return train_losses, val_losses


In [34]:
model = Transformer(
    embed_dim=512,
    vocab_size_src=tkz_en.get_vocab_size(),
    vocab_size_tgt=tkz_de.get_vocab_size(),
    key_dim=256,
    value_dim=256,
    num_heads=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    max_seq_len=100
)

In [ ]:
train_df, val_df = sample(df, 1000000, 0.1)
train_loader - s-s  \
    ]   

In [ ]:
# generate random embed_tensor
embed_tensor = torch.rand((500, 100))
vocab_size = 500
key_dim = 100
value_dim = 100
num_heads = 5
model = Transformer(100, vocab_size, embed_tensor, key_dim, value_dim, num_heads)

In [ ]:
x, y = torch.randint(0, vocab_size, (10, 100)), torch.randint(0, vocab_size, (10, 100))
print(x.shape, y.shape)

torch.Size([10, 100]) torch.Size([10, 100])


In [ ]:
model(x, y)

torch.Size([10, 100])